In [ ]:
import scipy.sparse as sp
import numpy as np
from topact.countdata import CountMatrix
from topact.classifier import SVCClassifier, train_from_countmatrix

# ── 加载 snRNA-seq（Seurat 导出的矩阵格式）──────────────────────
import scanpy as sc
adata = sc.read_10x_mtx("./data/scrna/")  # 或 read_h5ad

# 论文使用 Seurat 聚类后的 cell type 注释
# 对应 Extended Data Fig.1，共 23 个细胞类型（PCT, TAL, DCT, ENDO, POD 等）
labels = adata.obs["cell_type"].tolist()     # 你的注释列名
genes  = adata.var_names.tolist()
mtx    = sp.csr_matrix(adata.X)             # (n_cells, n_genes)

# ── 构建 TopACT CountMatrix 并训练 SVM ────────────────────────────
sc_data = CountMatrix(mtx, genes=genes)
sc_data.add_metadata("celltype", labels)

clf = SVCClassifier()          # 默认 RBF SVM + Platt scaling
train_from_countmatrix(clf, sc_data, "celltype")

# 保存分类器（避免重复训练）
import pickle
with open("./topact_clf.pkl", "wb") as f:
    pickle.dump(clf, f)

In [ ]:
import pandas as pd

# Xenium 输出目录包含 transcripts.csv.gz
# 每行是一条转录本记录
df_raw = pd.read_csv("./data/xenium/transcripts.csv.gz")
print(df_raw.columns)
# 通常包含：transcript_id, cell_id, overlaps_nucleus,
#           feature_name, x_location, y_location, z_location, qv

# ── 论文的预处理步骤（Methods 原文）────────────────────────────────
# 1. 只保留 Q-score ≥ 20 的转录本
df = df_raw[df_raw["qv"] >= 20].copy()

# 2. 只保留 snRNA-seq 中有的基因（特征对齐）
shared_genes = set(genes) & set(df["feature_name"].unique())
df = df[df["feature_name"].isin(shared_genes)]

# 3. 坐标已是 μm 单位，bin 到 1μm 整数格（论文：binned to 1μm resolution）
df["x"] = df["x_location"].round(0).astype(int)
df["y"] = df["y_location"].round(0).astype(int)

# 最终格式：每行 = 一条转录本，包含 gene, x, y, count(=1)
df_topact = df[["feature_name", "x", "y"]].copy()
df_topact.columns = ["gene", "x", "y"]
df_topact["counts"] = 1

print(f"转录本总数: {len(df_topact):,}")
print(f"共用基因数: {len(shared_genes)}")

In [ ]:
from topact import spatial
import numpy as np

# 构建 CountGrid（1μm 分辨率的空间表达格）
sd = spatial.CountGrid.from_coord_table(
    df_topact,
    genes=list(genes),       # 与训练分类器完全相同的基因列表
    count_col="counts",
    gene_col="gene",
)

print(f"空间网格尺寸: {sd.shape}")

# ── 多尺度分类（论文参数）────────────────────────────────────────────
# Methods: min_radius=2μm, max_radius=5μm, confidence=0.9
sd.classify_parallel(
    clf,
    min_scale=2,          # 最小聚合半径 2μm
    max_scale=5,          # 最大聚合半径 5μm
    num_proc=16,          # 并行进程数，根据你的CPU调整
    outfile="./topact_confidence.npy",
    verbose=True,
)

# ── 从置信矩阵提取最终细胞类型 ────────────────────────────────────
confidence_mtx = np.load("./topact_confidence.npy")
threshold = 0.9           # 论文使用 0.9

annotations = spatial.extract_image(confidence_mtx, threshold)
# annotations: 2D array，每个 1μm spot 的预测类别（整数）
# -1 表示在所有尺度下置信度都未达阈值

# 解码为细胞类型名称
class_names = clf.classes_
label_image = np.where(
    annotations >= 0,
    np.array(class_names)[np.clip(annotations, 0, None)],
    "unassigned"
)

In [ ]:
from topact import spatial
import numpy as np

# 构建 CountGrid（1μm 分辨率的空间表达格）
sd = spatial.CountGrid.from_coord_table(
    df_topact,
    genes=list(genes),       # 与训练分类器完全相同的基因列表
    count_col="counts",
    gene_col="gene",
)

print(f"空间网格尺寸: {sd.shape}")

# ── 多尺度分类（论文参数）────────────────────────────────────────────
# Methods: min_radius=2μm, max_radius=5μm, confidence=0.9
sd.classify_parallel(
    clf,
    min_scale=2,          # 最小聚合半径 2μm
    max_scale=5,          # 最大聚合半径 5μm
    num_proc=16,          # 并行进程数，根据你的CPU调整
    outfile="./topact_confidence.npy",
    verbose=True,
)

# ── 从置信矩阵提取最终细胞类型 ────────────────────────────────────
confidence_mtx = np.load("./topact_confidence.npy")
threshold = 0.9           # 论文使用 0.9

annotations = spatial.extract_image(confidence_mtx, threshold)
# annotations: 2D array，每个 1μm spot 的预测类别（整数）
# -1 表示在所有尺度下置信度都未达阈值

# 解码为细胞类型名称
class_names = clf.classes_
label_image = np.where(
    annotations >= 0,
    np.array(class_names)[np.clip(annotations, 0, None)],
    "unassigned"
)